In [1]:
!pip install torch triton

In [2]:
!nvidia-smi

Sat Mar  7 00:20:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import torch
import triton
import triton.language as tl

In [4]:
@triton.jit
def add_kernel(
    x_ptr,
    y_ptr,
    output_ptr,
    n_elements,
    BLOCK_SIZE: tl.constexpr
):
  # Nos estamos ubicando en el BLOCK de procesamiento
  pid = tl.program_id(axis=0)

  #  0  0  0  1  1  1
  # [1, 2, 1, 3 ,2 ,3]
  block_start = pid * BLOCK_SIZE

  #   0  0  0  1  1  1
  # [ 0, 1, 2, 3, 4, 5]
  offsets = block_start * tl.arange(0, BLOCK_SIZE)

  mask = offsets < n_elements

  x = tl.load(x_ptr + offsets, mask=mask)
  y = tl.load(y_ptr + offsets, mask=mask)

  # paralelizada automaticamente por triton
  output = x + y

  tl.store(output_ptr + offsets, output, mask=mask)

In [5]:
def run_add(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
  output = torch.empty_like(x)
  n_elements = output.numel() # numeral elements

  grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']),)
  add_kernel[grid](x, y, output, n_elements, BLOCK_SIZE=1024)
  return output

In [6]:
torch.manual_seed(42)

x = torch.randn(1000000, device='cuda')
y = torch.randn(1000000, device='cuda')

In [8]:
%%timeit
out_add = run_add(x, y)

31.8 µs ± 6.52 µs per loop (mean ± std. dev. of 7 runs, 10000 loops each)


In [13]:
@triton.jit
def softmax_kernel(
    output_ptr,
    input_ptr,
    input_row_strides,
    n_cols,
    BLOCK_SIZE: tl.constexpr
):
  pid = tl.program_id(0)
  # [M, N]
  row_str_ptr = input_ptr + pid * input_row_strides
  col_offsets = tl.arange(0, BLOCK_SIZE)

  inputs_ptr = row_str_ptr + col_offsets

  mask = col_offsets < n_cols

  row = tl.load(inputs_ptr, mask=mask, other=-float('inf'))

  row_minus_max = row - tl.max(row, axis=0)
  numerator = tl.exp(row_minus_max)
  denominator = tl.sum(numerator, axis=0) # <--------------- HORRIBLE HACERLA A MANO

  softmax_output = numerator / denominator

  output_ptrs = row_str_ptr + col_offsets

  tl.store(output_ptrs, softmax_output, mask=mask)

In [14]:
def run_softmax(x: torch.Tensor):
  output = torch.empty_like(x)
  M, N = x.shape # (M, N)

  BLOCK_SIZE = triton.next_power_of_2(N)

  grid = lambda meta: (M,)
  softmax_kernel[grid](output, x, x.stride(0), N, BLOCK_SIZE=BLOCK_SIZE)

In [15]:
x_soft = torch.randn(256, 128, device="cuda")

In [16]:
%%timeit
out_soft = run_softmax(x_soft)

87.6 µs ± 56.2 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [24]:
@triton.autotune(

)
@triton.jit
def matmul_kernel(
    a_ptr,
    b_ptr,
    c_ptr, # output
    M, N, K,
    stride_am,
    stride_ak,
    stride_bk,
    stride_bn,
    stride_cm,
    stride_cn,
    BLOCK_SIZE_M: tl.constexpr,
    BLOCK_SIZE_N: tl.constexpr,
    BLOCK_SIZE_K: tl.constexpr
):
  """
  GEMM (C = A x B)

  - carga los Tiles de A y B.
  - Acumular producots parciales en registros.
  - Minimizar el tráfico repetido a la HBM
  """
  pid_m = tl.program_id(0)
  pid_n = tl.program_id(1)

  # 1: Indices del title de salida (M, N) que procesa el programa.
  offs_am = (pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)) % M
  offs_bn = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N
  offs_k = tl.arange(0, BLOCK_SIZE_K)

  a_ptrs = a_ptr + (offs_am[:, None] * stride_am + offs_k[None, :] * stride_ak)
  b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)

  acc = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)

  for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
    a = tl.load(a_ptrs, mask=offs_k[None, :] < K - k * BLOCK_SIZE_K, other=0.0)
    b = tl.load(b_ptrs, mask=offs_k[:, None] < K - k * BLOCK_SIZE_K, other=0.0)

    acc += tl.dot(a, b)

    a_ptrs += BLOCK_SIZE_K * stride_ak
    b_ptrs += BLOCK_SIZE_K * stride_bk


  offs_cm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
  offs_cn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)

  c_ptrs = c_ptr + stride_cm * offs_cm[:, None] + stride_cn * offs_cn[None, :]

  mask = (offs_cm[:, None] < M) & (offs_cn[None, :] < N)
  tl.store(c_ptrs, acc, mask=mask)

In [25]:
def run_matmul(a: torch.Tensor, b: torch.Tensor):
  M, K = a.shape
  K, N = b.shape
  c = torch.empty((M, N), device="cuda", dtype=torch.float32)

  grid = lambda meta: (
      triton.cdiv(M, meta['BLOCK_SIZE_M']),
      triton.cdiv(N, meta['BLOCK_SIZE_N'])
  )

  matmul_kernel[grid](
      a, b, c,
      M, N, K,
      a.stride(0), a.stride(1),
      b.stride(0), b.stride(1),
      c.stride(0), c.stride(1),
      BLOCK_SIZE_M=64, BLOCK_SIZE_N=64, BLOCK_SIZE_K=32
  )

  return c

In [29]:
A = torch.randn(10000, 12000, device="cuda", dtype=torch.float32)
B = torch.randn(12000, 24000, device="cuda", dtype=torch.float32)

In [ ]:
%%timeit
out_matmul = run_matmul(A, B)